# Black-Litterman Backtest Example

This notebook demonstrates the backtest logic clearly.

`backtest.ipynb` is a Jupyter notebook file, so it cannot be imported directly like a Python module. To make the example runnable, the core equations and backtest functions are imported from `backtest_utils.py`, which is extracted from the notebook logic.

## 1. Setup

Import the reusable backtest functions and classes.

In [ ]:
import numpy as np
import pandas as pd

from backtest_utils import (
    BlackLitterman,
    PerformanceMetrics,
    RollingWindowBacktest,
    get_standard_params,
    get_standard_views,
    run_experiment,
    run_experiment_improve_omega_only,
    run_experiment_improve_sigma_only,
)

## 2. Load Return Data

In [ ]:
returns = pd.read_csv('etf_returns.csv', index_col=0, parse_dates=True)
returns.index = pd.to_datetime(returns.index)

print(f'Data shape: {returns.shape}')
print(f'Assets: {list(returns.columns)}')
print(f'Date range: {returns.index.min().date()} to {returns.index.max().date()}')
returns.head()

## 3. One Rebalance Step: Show the Core BL Logic

This cell shows the exact logic inside one rebalance date:

1. Use the training window to estimate `mu` and `sigma`.
2. Generate views `P`, `q`, and `omega`.
3. Use Black-Litterman to get posterior return and covariance.
4. Optimize the portfolio weights.

In [ ]:
train_window = 252 * 3
train_data = returns.iloc[:train_window]

mu, sigma = get_standard_params(train_data)
P, q, omega = get_standard_views(train_data)
market_weights = np.full(returns.shape[1], 1 / returns.shape[1])

bl = BlackLitterman(market_weights=market_weights, cov_matrix=sigma, tau=0.05)
mu_post, sigma_post = bl.update_with_views(P, q, omega)
weights, _, _ = bl.optimize_with_views(P, q, omega)

display(pd.Series(mu, index=returns.columns, name='prior_mu_annualized'))
display(pd.DataFrame(P, columns=returns.columns, index=['view_1', 'view_2']))
display(pd.Series(q, index=['view_1', 'view_2'], name='view_returns'))
display(pd.Series(mu_post, index=returns.columns, name='posterior_mu'))
display(pd.Series(weights, index=returns.columns, name='optimized_weights'))

## 4. Full Rolling Backtest

Now run the same logic repeatedly in a rolling window backtest.

For each rebalance date, the engine does:

1. Train on the previous `train_window` returns.
2. Build BL views from the training data.
3. Compute new weights.
4. Apply those weights until the next rebalance date.

In [ ]:
backtest = RollingWindowBacktest(
    returns_df=returns,
    train_window=252 * 3,
    rebalance_freq=252,
)

portfolio_values, portfolio_returns, weights_history = backtest.backtest_bl(
    get_params_func=get_standard_params,
    get_views_func=get_standard_views,
    tau=0.05,
)

metrics = PerformanceMetrics(
    portfolio_values=portfolio_values,
    returns=portfolio_returns,
    weights_history=weights_history,
).get_metrics()

pd.Series(metrics, name='standard_bl_metrics')

## 5. Example: Improve Only Sigma

This example keeps the BL views unchanged and only changes the covariance matrix estimate.

In [ ]:
demo_train = returns.iloc[:252 * 3]
standard_sigma = demo_train.cov().values
improved_sigma_demo = standard_sigma * 0.95

comparison_sigma, results_sigma = run_experiment_improve_sigma_only(
    returns_data=returns,
    improved_sigma=improved_sigma_demo,
    tau=0.05,
    train_window=252 * 3,
    rebalance_freq=252,
    plot=True,
)

comparison_sigma

## 6. Example: Improve Only Omega

This example keeps Sigma, `P`, and `q` unchanged and only changes the confidence matrix `omega`.

In [ ]:
demo_train = returns.iloc[:252 * 3]
P_demo, q_demo, _ = get_standard_views(demo_train)
demo_sigma = demo_train.cov().values
view_variance = np.array([np.dot(P_demo[i], np.dot(demo_sigma, P_demo[i])) for i in range(len(P_demo))])
standard_omega_demo = np.diag(view_variance * 0.05)
improved_omega_demo = standard_omega_demo * 0.7

comparison_omega, results_omega = run_experiment_improve_omega_only(
    returns_data=returns,
    improved_omega=improved_omega_demo,
    tau=0.05,
    train_window=252 * 3,
    rebalance_freq=252,
    plot=True,
)

comparison_omega

## 7. Example: Improve Sigma and Omega Together

This is the most general experiment interface.

In [ ]:
comparison_both, results_both = run_experiment(
    returns_data=returns,
    custom_sigma=improved_sigma_demo,
    custom_omega=improved_omega_demo,
    tau=0.05,
    train_window=252 * 3,
    rebalance_freq=252,
    plot=True,
)

comparison_both